In [340]:
%reset -f
import os
import glob
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import numpy as np

datadir = "/home/kalfasyan/data/images/sticky_plates/"
created_data_path = f"{datadir}/created_data/"
years = ['2019','2020']

messed_up_index_plates = ["beauvech_w38_A_F10_51 mm_ISO160_1-15 s",
                        "beauvech_w39_A_F10_51 mm_ISO160_1-15 s",
                        "beauvech_w39_C_F10_51 mm_ISO160_1-15 s",
                        "brainelal_w38_A_F10_51 mm_ISO160_1-15 s",
                        "brainelal_w38_C_F10_51 mm_ISO160_1-15 s",
                        "brainelal_w38_B_F10_51 mm_ISO160_1-15 s",
                        "brainelal_w39_A_F10_51 mm_ISO160_1-15 s",
                          "brainelal_w39_B_F10_51 mm_ISO160_1-15 s",]

standard_cols = ['name plate', 'index']
labview_cols = ['Center of Mass X.1', 'Center of Mass Y.1', 'Bounding Rect Left.1',
       'Bounding Rect Top.1', 'Bounding Rect Right.1',
       'Bounding Rect Bottom.1', 'Equivalent Ellipse Major Axis.1',
       'Equivalent Ellipse Minor Axis.1', 'Area.1', 'Convex Hull Area.1',
       'Orientation.1', 'Ratio of Equivalent Ellipse Axes.1',
       'Ratio of Equivalent Rect Sides.1', 'Elongation Factor.1',
       'Compactness Factor.1', 'Heywood Circularity Factor.1', 'Type Factor.1',
       'R', 'G', 'B']
cols_in_order = standard_cols + labview_cols + ['Klasse']

In [341]:
%%time
for year in years:
    if not os.path.isfile(f"{created_data_path}/labview_results_{year}.csv"):        
        path = f'{datadir}/{year}/'
        labview_results_list = glob.glob(path + "/**/*.xlsx", recursive = True)
        labview_results_list = [i for i in labview_results_list if not any(x in i for x in ['annotations', 'results_python_ioannis'])]
        all_lbv_res = []
        for res in tqdm(labview_results_list):
            all_lbv_res.append(pd.read_excel(res))
        labview_results = pd.concat(all_lbv_res)
        labview_results.to_csv(f"{created_data_path}/labview_results_{year}.csv")

CPU times: user 89 µs, sys: 22 µs, total: 111 µs
Wall time: 83.2 µs


In [342]:
dataframes = []

for year in years:
    print(f"\n-- Processing expert labels for year: {year} --\n")
    labels_dir = f"{created_data_path}/expert_labels/{year}"
    if not os.path.isdir(labels_dir):
        os.mkdir(labels_dir)

    xlsx_files = [fname for fname in glob.iglob(labels_dir + '/**/*.xlsx', recursive=True)]
    assert len(xlsx_files), "No expert labels found. (excel files provided by a Proefcentrum)"

    print(f'Number of excel annotation files found: {len(xlsx_files)}')
    wanted_columns_set = set(['name plate', 'index', 'Klasse', 'klasse'])
    df_labeldata = []
    for f in xlsx_files:
        print(f"Processing annotation file: {f.split('/')[-1]}")
        if f.endswith('w00.xlsx'):
            print(f"Skipping file: {f.split('/')[-1]}")
            continue

        sub = pd.read_excel(f, engine='openpyxl')
        
        assert sub.iloc[:,1].name == 'index'
        assert sub.iloc[:,1].iloc[0] == 0., 'Check if excel file index starts with 1 instead of 0.'        
        try:
            df = sub[standard_cols + labview_cols + ['Klasse']] #sub[list(wanted_columns_set.intersection(sub.columns))]
        except:
            all_merged = []
            for i, (q, dftmp) in enumerate(sub.groupby('name plate')):
                labview_results = pd.read_csv(f"{created_data_path}labview_results_{year}.csv", index_col='Unnamed: 0')
                lv_res = labview_results[labview_results['name plate'] == dftmp['name plate'].unique()[0]][standard_cols + labview_cols]
                df_merged = pd.merge(dftmp, lv_res, on=['name plate','index'])
                all_merged.append(df_merged)
            df = pd.concat(all_merged)
            df = df[cols_in_order]
        assert len(df.columns) == 23, 'Check excel file columns.'
#         df.columns = map(str.lower, df.columns)
        df.rename(columns={'name plate': 'platename', 'Klasse': 'class', 'index': 'bbox_idx'}, inplace=True)

        problematic_inds = []
        fixed_subdfs = []
        for i,q in df.groupby('platename'):

            if q.bbox_idx.iloc[0] != 0:
                print(f"{q.bbox_idx.iloc[0]} found instead of 0 in first index of plate. Discarding it")
                problematic_inds.append(q.iloc[0].name)

            if i in messed_up_index_plates:
            #    q.bbox_idx = q.bbox_idx + 1.
               q['class'] = q['class'].shift(+1)
            fixed_subdfs.append(q)

        df = pd.concat(fixed_subdfs, axis=0)
        df.drop(problematic_inds, axis=0, inplace=True)
        assert df.bbox_idx.isna().sum() == 0
        df_labeldata.append(df)
#         break

    sub = pd.concat(df_labeldata, axis=0)
    sub['class'] = sub['class'].apply(lambda x: str(x).replace(" ","").lower())
    # sub['class'] = sub['class'].apply(lambda x: str(x).replace("2",""))
    # sub['class'] = sub['class'].apply(lambda x: str(x).replace("3",""))
    # sub['class'] = sub['class'].apply(lambda x: str(x).replace("4",""))
    dataframes.append(sub)
    
    break

df = pd.concat(dataframes, axis=0)
le = LabelEncoder()
df['class_encoded'] = le.fit_transform(df['class'].tolist())

path_annotations = f'{created_data_path}/annotations/'
assert os.path.isdir(path_annotations), "Annotations path not created."

mapped = dict(zip(le.transform(le.classes_), le.classes_))
# # Saving class mapping to use as yolo annotation classes
# pd.Series(mapped).to_csv(f'{path_annotations}/classes.txt', sep=' ')
# # Saving class mapping to use when processing each plate
# df.to_csv(f'{created_data_path}/class_mapping.csv')              


-- Processing expert labels for year: 2019 --

Number of excel annotation files found: 14
Processing annotation file: results_20200212_W40 .xlsx


/home/kalfasyan/anaconda3/envs/objdetect/lib/python3.7/site-packages/pandas/core/frame.py:4133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


Processing annotation file: results_datageleplaten_w00.xlsx
Skipping file: results_datageleplaten_w00.xlsx
Processing annotation file: results_20190730_20190806.xlsx
Processing annotation file: results_2019_annotations_yannis.xlsx
Processing annotation file: results_20190821.xlsx
Processing annotation file: results_20191004_W38_en_W39.xlsx


/home/kalfasyan/anaconda3/envs/objdetect/lib/python3.7/site-packages/ipykernel_launcher.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


nan found instead of 0 in first index of plate. Discarding it
Processing annotation file: results_20191001_W35.xlsx
Processing annotation file: results_20200129_W41 .xlsx
Processing annotation file: results_20190809_ W31.xlsx
Processing annotation file: results_20191001_W36.xlsx
Processing annotation file: results_may_june.xlsx
Processing annotation file: results_20191001_W34.xlsx
Processing annotation file: results_20191001_W37.xlsx
Processing annotation file: results_20180806_W29.xlsx


In [343]:
df.to_csv(f"{created_data_path}/df_all_plates_info.csv", index=False)

In [344]:
from utils import clean_folder
created_data_path = f'{datadir}/created_data'
clean = True
# CREATING NECESSARY DIRECTORIES FOR THE PROJECT
path_annotations = f'{created_data_path}/annotations/'
path_images = f'{created_data_path}/images/'
path_voc_annotations = f'{created_data_path}/voc_annotations/'
path_crops_export = f'{created_data_path}/crops_export/'
path_images_augmented = f'{created_data_path}/images_augmented/'
path_weights = f'{created_data_path}/weights/'
path_logs = f'{created_data_path}/logs/'
for path in [created_data_path, path_annotations, path_images, path_voc_annotations, 
            path_crops_export, path_weights, path_logs, path_images_augmented]:
    if not os.path.isdir(path):
        os.mkdir(path)	

if clean:
    print(f'Cleaning directories..')
    clean_folder(path_annotations)
    clean_folder(path_images)
    clean_folder(path_voc_annotations)
    os.system(f'rm -rf {path_crops_export}*')
    os.system(f'rm -rf {path_images_augmented}*')
    os.system(f'rm {created_data_path}/class_mapping.csv')
assert len(os.listdir(path_crops_export)) <= 0, "Wrong"

Cleaning directories..


In [350]:
df = pd.read_csv(f"{created_data_path}/df_all_plates_info.csv")
df = df.loc[df.platename.apply(lambda x: 'yellow' not in x)] # discarding yellow plates
df['Bounding Rect Bottom.1'] = df['Bounding Rect Bottom.1'] / 1000000
df['Bounding Rect Left.1'] = df['Bounding Rect Left.1'] / 1000000 
df['Bounding Rect Top.1'] = df['Bounding Rect Top.1'] / 1000000
df['Bounding Rect Right.1'] = df['Bounding Rect Right.1'] / 1000000

In [380]:
%%time
from utils import get_plate_names
plates = []
for y in years:
    y_plates = get_plate_names(y, base_dir=datadir)
    plates += y_plates
    print(f"Number of plates: {len(y_plates)} for year: {y}")
    
plates_series = pd.Series(plates).apply(lambda x: x.split('/')[-1][:-4]).tolist()

import imagesize

plate_dims = pd.Series(plates).apply(lambda x: imagesize.get(x))
w,h = zip(*plate_dims)

df_plateinfo = pd.DataFrame({"platefilenames": plates, "platenames": plates_series, "w":w, "h":h})
df_plateinfo.set_index('platenames', inplace=True)

df['W'] = df.platename.apply(lambda x: df_plateinfo.loc[x].w if x in plates_series else None)
df['H'] = df.platename.apply(lambda x: df_plateinfo.loc[x].h if x in plates_series else None)
df['platefilenames'] = df.platename.apply(lambda x: df_plateinfo.loc[x].platefilenames if x in plates_series else None)
df['year'] = df.platefilenames.apply(lambda x: x.split('/')[len(datadir.split('/'))-1] if isinstance(x,str) else None)

df['yolo_x'] = np.abs(df['Bounding Rect Right.1'] - np.abs(df['Bounding Rect Left.1'] - df['Bounding Rect Right.1']) /2) / df['W']
df['yolo_y'] = np.abs(df['Bounding Rect Bottom.1'] - np.abs(df['Bounding Rect Top.1'] - df['Bounding Rect Bottom.1']) /2) / df['H']
df['width'] = 150
df['height'] = 150
df['yolo_width'] = pd.concat([df['width'], df['height']], axis=1).max(axis=1) / df['W']
df['yolo_height'] = pd.concat([df['width'], df['height']], axis=1).max(axis=1) / df['H']

Number of plates: 398 for year: 2019
Number of plates: 245 for year: 2020
CPU times: user 19.4 s, sys: 116 ms, total: 19.6 s
Wall time: 19.5 s


In [387]:
def save_insect_crops(specifications, path_crops):
    import cv2

    for _, row in specifications.iterrows():
        H = row.H
        W = row.W
        left  = int((row.yolo_x-row.yolo_width/2.)*W)
        right = int((row.yolo_x+row.yolo_width/2.)*W)
        top   = int((row.yolo_y-row.yolo_height/2.)*H)
        bot   = int((row.yolo_y+row.yolo_height/2.)*H)

        if(left < 0): left = 0;
        if(right > W-1): right = W-1;
        if(top < 0): top = 0;
        if(bot > H-1): bot = H-1;

        print(f"left: {left}, right: {right}, top: {top}, bot: {bot}, \t W:{W}, H:{H}")
        plate_img = cv2.imread(row.platefilenames)
#         print(bot)
#         print(type(bot))
        crop = plate_img[top:bot, left:right]

        savepath = f"{path_crops}/{row['class']}/"
        if not os.path.isdir(savepath):
            os.makedirs(savepath)
        cv2.imwrite(f"{savepath}/{row.year}_{row.platename}_{row.bbox_idx}.jpg", crop)

In [388]:
df

,platename,bbox_idx,Center of Mass X.1,Center of Mass Y.1,Bounding Rect Left.1,Bounding Rect Top.1,Bounding Rect Right.1,Bounding Rect Bottom.1,Equivalent Ellipse Major Axis.1,Equivalent Ellipse Minor Axis.1,...,W,H,platefilenames,yolo_x,yolo_y,width,height,yolo_width,yolo_height,year
0,beauvech_w40_A_F10_51 mm_ISO160_1-15 s,0.0,2.961977e+09,1.271136e+08,2942.0,118.0,2980.0,134.0,50464857.0,7770908.0,...,NaN,NaN,None,NaN,NaN,150,150,NaN,NaN,None
1,beauvech_w40_A_F10_51 mm_ISO160_1-15 s,1.0,2.214756e+09,1.285753e+08,2185.0,123.0,2248.0,135.0,71786945.0,7768528.0,...,NaN,NaN,None,NaN,NaN,150,150,NaN,NaN,None
2,beauvech_w40_A_F10_51 mm_ISO160_1-15 s,2.0,5.457518e+09,2.786412e+08,5438.0,266.0,5478.0,296.0,46915333.0,16337734.0,...,NaN,NaN,None,NaN,NaN,150,150,NaN,NaN,None
3,beauvech_w40_A_F10_51 mm_ISO160_1-15 s,3.0,5.627737e+09,3.821620e+08,5612.0,362.0,5642.0,403.0,57165750.0,14989574.0,...,NaN,NaN,None,NaN,NaN,150,150,NaN,NaN,None
4,beauvech_w40_A_F10_51 mm_ISO160_1-15 s,4.0,5.443604e+09,3.936130e+08,5407.0,378.0,5467.0,413.0,69128133.0,21273418.0,...,NaN,NaN,None,NaN,NaN,150,150,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80125,her_262719_16_27_F11_51 mm_ISO160_1-15 s,59.0,1.504712e+09,3.604246e+09,1487.0,3584.0,1524.0,3625.0,49812075.0,17330264.0,...,6436.0,4331.0,/home/kalfasyan/data/images/sticky_plates/2019...,0.233919,0.832256,150,150,0.023306,0.034634,2019
80126,her_262719_16_27_F11_51 mm_ISO160_1-15 s,60.0,1.561129e+09,3.688538e+09,1554.0,3671.0,1571.0,3715.0,56815715.0,7664216.0,...,6436.0,4331.0,/home/kalfasyan/data/images/sticky_plates/2019...,0.242775,0.852690,150,150,0.023306,0.034634,2019
80127,her_262719_16_27_F11_51 mm_ISO160_1-15 s,61.0,2.536942e+09,3.835381e+09,2519.0,3821.0,2563.0,3855.0,52720117.0,17050553.0,...,6436.0,4331.0,/home/kalfasyan/data/images/sticky_plates/2019...,0.394810,0.886169,150,150,0.023306,0.034634,2019
80128,her_262719_16_27_F11_51 mm_ISO160_1-15 s,62.0,4.913449e+09,3.838719e+09,4899.0,3831.0,4928.0,3845.0,30428080.0,12218515.0,...,6436.0,4331.0,/home/kalfasyan/data/images/sticky_plates/2019...,0.763440,0.886169,150,150,0.023306,0.034634,2019


In [389]:
save_insect_crops(df.dropna(), path_crops_export)

left: 4935, right: 5085, top: 114, bot: 264, 	 W:6278.0, H:4183.0
left: 654, right: 804, top: 124, bot: 274, 	 W:6278.0, H:4183.0
left: 5444, right: 5594, top: 132, bot: 283, 	 W:6278.0, H:4183.0
left: 4767, right: 4917, top: 176, bot: 325, 	 W:6278.0, H:4183.0
left: 2188, right: 2337, top: 235, bot: 385, 	 W:6278.0, H:4183.0
left: 771, right: 921, top: 164, bot: 314, 	 W:6278.0, H:4183.0
left: 5236, right: 5386, top: 198, bot: 348, 	 W:6278.0, H:4183.0
left: 4818, right: 4968, top: 178, bot: 328, 	 W:6278.0, H:4183.0
left: 3814, right: 3964, top: 202, bot: 352, 	 W:6278.0, H:4183.0
left: 4014, right: 4164, top: 207, bot: 357, 	 W:6278.0, H:4183.0
left: 3260, right: 3410, top: 205, bot: 356, 	 W:6278.0, H:4183.0
left: 2544, right: 2694, top: 212, bot: 363, 	 W:6278.0, H:4183.0
left: 4878, right: 5028, top: 244, bot: 395, 	 W:6278.0, H:4183.0
left: 857, right: 1007, top: 215, bot: 365, 	 W:6278.0, H:4183.0
left: 2424, right: 2574, top: 224, bot: 374, 	 W:6278.0, H:4183.0
left: 3541, rig

KeyboardInterrupt: 